# AQDrop Job Submission and Retrieval Example

This notebook demonstrates how to submit a simple Bell-state quantum circuit to an AQDrop queue and then retrieve the results.

In [15]:
# install letest version - if needed
#!  pip install --upgrade "aqdrop"

In [16]:
! pip list |grep aqdrop

aqdrop                    0.13


In [17]:
# IMPORTS
import os
from pprint import pprint
from qiskit import QuantumCircuit
from aqdrop import AqdropClient

In [18]:
# list my last jobs
! aqdrop job_list |tail

1058   jan_u        X6Y3         success     2026-04-30 13:31:57 PDT
1091   jan_u        noisy        cancelled   2026-05-07 00:50:45 PDT
1092   jan_u        noisy        cancelled   2026-05-07 00:51:01 PDT
1093   jan_u        noisy        cancelled   2026-05-07 00:27:10 PDT
1094   jan_u        noisy        cancelled   2026-05-07 00:27:44 PDT
1095   jan_u        noisy        cancelled   2026-05-07 00:29:44 PDT
1098   jan_u        noisy        success     2026-05-07 00:39:07 PDT
1099   jan_u        noisy        success     2026-05-07 00:44:31 PDT
1100   jan_u        noisy        success     2026-05-07 00:59:53 PDT
1101   jan_u        noisy        declined    2026-05-07 01:03:36 PDT


In [19]:
# list  avaliable queues
! aqdrop queue_list 

## Initialize Client

In [20]:
# WARN - will reveal your secret if activated
# ! env |grep AQDROP

In [21]:
# note that AqdropUser automatically instantiates from the environment variables:
# NERSC_OIDC_TOKEN, AQDROP_HOSTNAME

from AqdropUser import AqdropUser
user = AqdropUser(verb=1)

## Initialize Circuit

We test a circuit that makes a Bell state. This circuit is a simple entanglement between qubits 0 and 1.

In [22]:
def circ_bell():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure(0, 0)
    qc.measure(1, 1)
    return qc

qc = circ_bell()
print("Bell State Circuit:")
print(qc.draw())

Bell State Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [23]:
# Assemble job input
job_meta = {
    "shots": [1024], 
    "comment": "Notebook Bell State Job", 
    "queue_name": 'noisy',
    "pref_qubits" : None
}

# 2) assemble job_input from circuits + metadata
user.assemble_job_input([qc], job_meta)
job_id = user.push_job_input()

assembled job_input for 1 circuits, queue=noisy
idx  num_qubits  num_shots
  0           2  1024
pushed job_input; assigned job_id=1102
   ./job_retrieve.py --id 1102


## Retrieving the Job

Now we will retrieve the results for the job we just submitted. Note that depending on the queue, the job might still be `queued`.

In [24]:
! aqdrop job_list |tail

1091   jan_u        noisy        cancelled   2026-05-07 00:50:45 PDT
1092   jan_u        noisy        cancelled   2026-05-07 00:51:01 PDT
1093   jan_u        noisy        cancelled   2026-05-07 00:27:10 PDT
1094   jan_u        noisy        cancelled   2026-05-07 00:27:44 PDT
1095   jan_u        noisy        cancelled   2026-05-07 00:29:44 PDT
1098   jan_u        noisy        success     2026-05-07 00:39:07 PDT
1099   jan_u        noisy        success     2026-05-07 00:44:31 PDT
1100   jan_u        noisy        success     2026-05-07 00:59:53 PDT
1101   jan_u        noisy        declined    2026-05-07 01:03:36 PDT
1102   jan_u        noisy        queued      2026-05-07 01:04:08 PDT


In [25]:
# cancel job if   you changed your mind
#! aqdrop job_cancel --id 1092

In [32]:
# Retrieve the job using the ID from the previous step
job = user.pull_job(job_id)
print(f"Job Status: {job['status']}")

pulled job: {'id': 1102, 'owner_name': 'jan_u', 'queue_name': 'noisy', 'status': 'success'}
Job Status: success


In [33]:
# Parse the job into its components
circL, inputMD, output, transpL = user.parse_job()

if job['status'] == "success":
    user.print_shot_summary()
    print(f"Total Execution Time: {output['tot_exec_time']:.1f} sec")
    print(f"Received Total Shots: {output['tot_shots']}")
    print(f"Calibration Version: {output['calib_ver']}")
    print(f"Execution Date: {output['exec_date']}")
    print(f'\ncounts: {output['counts']}\n')
else:
    print("Results are not available yet or the job failed.")

parsed job: 1 input circuits, output present, 1 transpiled circuits
shots asked=[1024]
received=[1024]
Total Execution Time: 0.9 sec
Received Total Shots: 1024
Calibration Version: fake_torino
Execution Date: 20260507_010449_PDT

counts: [{'01': 12, '10': 14, '00': 512, '11': 486}]



In [34]:
# Display circuits and transpiled circuits
print(f"Packed circuits: {len(circL)}, total requested shots: {sum(inputMD['shots'])}")

for idx, qc in enumerate(circL):
    print(f"\nCircuit {idx}:")
    print(qc)
    if transpL:
        print(f"\nTranspiled Circuit {idx}:")
        print(transpL[idx])

Packed circuits: 1, total requested shots: 1024

Circuit 0:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Transpiled Circuit 0:
global phase: 3π/4
          ┌─────────┐┌────┐ ┌───────┐    ┌────┐┌─────────┐┌─┐
q_1 -> 65 ┤ Rz(π/2) ├┤ √X ├─┤ Rz(π) ├──■─┤ √X ├┤ Rz(π/2) ├┤M├
          ├─────────┤├────┤┌┴───────┴┐ │ └┬─┬─┘└─────────┘└╥┘
q_0 -> 66 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─■──┤M├──────────────╫─
          └─────────┘└────┘└─────────┘    └╥┘              ║ 
     c: 2/═════════════════════════════════╩═══════════════╩═
                                           0               1 
